# PaddleOCR — Detection + Direction Classification + Recognition Pipeline

In [ ]:
!pip install -q paddlepaddle paddleocr pillow numpy

In [ ]:
import reimport numpy as npfrom PIL import Image, ImageDraw, ImageFontfrom paddleocr import PaddleOCR

In [ ]:
DATASET_PATH = ""GROUND_TRUTH_TEXT = "PaddleOCR detects rotates and recognizes text in one pipeline."LANGUAGE = "en"

In [ ]:
def create_sample_image(text, width=900, height=220, font_size=28):    image = Image.new("RGB", (width, height), color=(255, 255, 255))    draw = ImageDraw.Draw(image)    try:        font = ImageFont.truetype("DejaVuSans.ttf", font_size)    except IOError:        font = ImageFont.load_default()    draw.text((20, height // 2 - font_size), text, fill=(0, 0, 0), font=font)    return imagedef load_image(dataset_path, fallback_text):    if dataset_path:        return Image.open(dataset_path).convert("RGB")    return create_sample_image(fallback_text)sample_image = load_image(DATASET_PATH, GROUND_TRUTH_TEXT)sample_image

In [ ]:
ocr_engine = PaddleOCR(use_angle_cls=True, lang=LANGUAGE, show_log=False)

In [ ]:
image_array = np.array(sample_image)results = ocr_engine.ocr(image_array, cls=True)

In [ ]:
detections = results[0] if results else []for box, (text, score) in detections:    print(text, score, box)

In [ ]:
def draw_ocr_boxes(image, detections):    boxed = image.convert("RGB").copy()    draw = ImageDraw.Draw(boxed)    for box, (text, score) in detections:        polygon = [tuple(point) for point in box]        draw.line(polygon + [polygon[0]], fill=(255, 0, 0), width=2)    return boxedboxed_image = draw_ocr_boxes(sample_image, detections)boxed_image

In [ ]:
def merge_recognized_text(detections):    sorted_detections = sorted(detections, key=lambda d: d[0][0][1])    return " ".join(text for _, (text, score) in sorted_detections)recognized_text = merge_recognized_text(detections)print(recognized_text)

In [ ]:
def normalize_text(text, case_sensitive=False):    if not case_sensitive:        text = text.lower()    text = re.sub(r"\s+", " ", text).strip()    return list(text)def edit_distance_ops(ref_chars, hyp_chars):    n, m = len(ref_chars), len(hyp_chars)    dp = [[0] * (m + 1) for _ in range(n + 1)]    for i in range(n + 1):        dp[i][0] = i    for j in range(m + 1):        dp[0][j] = j    for i in range(1, n + 1):        for j in range(1, m + 1):            if ref_chars[i - 1] == hyp_chars[j - 1]:                dp[i][j] = dp[i - 1][j - 1]            else:                dp[i][j] = 1 + min(dp[i - 1][j - 1], dp[i][j - 1], dp[i - 1][j])    return dp[n][m]def cer(reference, hypothesis):    ref_chars = normalize_text(reference)    hyp_chars = normalize_text(hypothesis)    distance = edit_distance_ops(ref_chars, hyp_chars)    n = len(ref_chars) if len(ref_chars) > 0 else 1    return distance / nprint("cer:", cer(GROUND_TRUTH_TEXT, recognized_text))